![imagen](./img/ejercicios.png)

# INFORMACIÓN DEL PROYECTO

### TÍTULO

Análisis Exploratorio de Disruptores Endocrinos en la Alimentación Española: Más allá del Nutri-Score y el Eco-Score.

### TEMA

Evaluación la correlación entre el grado de procesamiento de los alimentos no-frescos en España y los posibles disruptores endocrinos (DEs).

### HIPÓTESIS

**Hipótesis 1:** "En los productos de alimentación disponibles en España, la exposición a potenciales disruptores endocrinos (DEs) no se distribuye aleatoriamente, sino que se concentra en productos con mayor grado de procesamiento (NOVA 4)."

**Hipótesis 2:** "En los productos de alimentación disponibles en España, sxiste una correlación positiva y estadísticamente significativa entre un peor perfil nutricional (NutriScore D/E) y una mayor exposición a aditivos con sospecha de disrupción endocrina."

**Hipótesis 3:** "Un buen impacto ambiental (EnvironmentalScore A o B) no garantiza una baja exposición a disruptores endocrinos. Existen categorías de productos donde la sostenibilidad del envase o el origen local enmascaran una alta carga de aditivos sintéticos, creando una falsa sensación de seguridad integral."

**Hipótesis 4:** "Los productos que ostentan *claims* frontales como 'Light', 'Zero', 'Fitness', 'Sin azúcares añadidos' o 'vegano' no reducen su exposición a disruptores endocrinos, y en algunos casos la aumentan, al sustituir ingredientes tradicionales por edulcorantes, colorantes o conservantes sintéticos de la lista de vigilancia."

## OBTENCIÓN DE LOS DATOS

### DATASETS Y FUENTES ALTERNATIVAS DE DATOS

Incluye aquí una vista del dataset o datasets de los que partirás para poder evaluar tu hipótesis. <br>
También incluye el origen de estos datos y su fuente.

* Datos descargados de https://es.openfoodfacts.org/data el 29 de octubre de 2026. He cargado sólo 50 filas porque son muchas si no para la prueba.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', 500)

#### Dataset productos de alimentación en España

In [2]:
# 1. Definimos la ruta correcta (con 'r')
ruta = r"C:\Users\alexa\Desktop\Denetik\CURSOS\Data-Science-BBKbootcamp\Carpeta-trabajo\Proyectos\Proyectos_individuales\2-EDA\datos\en.openfoodfacts.org.products.csv.gz"

# 2. Inicializamos una lista vacía y definimos nuestro objetivo
lista_chunks_espana = []
objetivo_filas = 50  # Queremos exactamente 50 filas de España

print("Iniciando la lectura por bloques... Esto será mucho más rápido ahora.")

# 3. Leemos el archivo en bloques de 50,000 filas. 
# ¡Hemos eliminado 'usecols' para que se carguen TODAS las columnas!
bloques = pd.read_csv(ruta, sep='\t', chunksize=50000, low_memory=False)

for i, chunk in enumerate(bloques):
    # Filtramos el bloque actual: buscamos filas donde 'countries_en' contenga 'Spain'
    chunk_filtrado = chunk[chunk['countries_en'].str.contains('Spain', case=False, na=False)]
    
    # Si el bloque contiene productos de España, lo añadimos a la lista
    if not chunk_filtrado.empty:
        lista_chunks_espana.append(chunk_filtrado)
        
        # Comprobamos si ya hemos alcanzado o superado las 20 filas
        filas_acumuladas = sum(len(df) for df in lista_chunks_espana)
        if filas_acumuladas >= objetivo_filas:
            print(f"✅ Objetivo de {objetivo_filas} filas alcanzado en el bloque {i}. Deteniendo la lectura para ahorrar recursos.")
            break  # <-- Esto detiene el bucle y deja de leer el archivo
    
    # Imprime un contador para saber que el código sigue vivo (cada 10 bloques)
    if i % 10 == 0:
        print(f"Bloques procesados: {i}...")

# 4. Concatenamos todos los trozos filtrados en un único DataFrame final
df_espana = pd.concat(lista_chunks_espana, ignore_index=True)

# 5. Nos aseguramos de quedarnos con exactamente 20 filas 
# (por si el último bloque procesado nos dejó con 25, por ejemplo)
df_espana = df_espana.head(objetivo_filas)

print(f"\n¡Listo! Se ha generado tu DataFrame con {df_espana.shape[0]} filas y {df_espana.shape[1]} columnas.")

Iniciando la lectura por bloques... Esto será mucho más rápido ahora.
✅ Objetivo de 50 filas alcanzado en el bloque 0. Deteniendo la lectura para ahorrar recursos.

¡Listo! Se ha generado tu DataFrame con 50 filas y 211 columnas.


In [3]:
df_espana.columns

Index(['code', 'url', 'creator', 'created_t', 'created_datetime',
       'last_modified_t', 'last_modified_datetime', 'last_modified_by',
       'last_updated_t', 'last_updated_datetime',
       ...
       'choline_100g', 'phylloquinone_100g', 'beta-glucan_100g',
       'inositol_100g', 'carnitine_100g', 'sulphate_100g', 'nitrate_100g',
       'acidity_100g', 'carbohydrates-total_100g', 'water_100g'],
      dtype='str', length=211)

In [4]:
# Estoy eligiendo las columnas que me interesan para después seleccionarlas del archivo 
# .csv.gz y que así tarde menos en cargar todos los datos
columnas_interes_alimentacion =  ["url", "created_datetime", "last_modified_datetime", 
                     "product_name", "generic_name", "quantity", "brands", 
                     "categories_tags", "categories_en", "categories",
                     "labels", "labels_tags", "labels_en", "stores", "countries_en",
                     "ingredients_text", "ingredients_tags", "ingredients_analysis_tags",
                     "nutriscore_score", "nutriscore_grade", "nova_group", 
                     "food_groups", "food_groups_tags", "food_groups_en",
                     "environmental_score_score", "environmental_score_grade", "allergens_en",
                     "additives_tags", "additives_en", "packaging", "packaging_tags", "brand_owner",
                     "manufacturing_places_tags", "origins_en", "packaging_en"]

len(columnas_interes_alimentacion)

35

In [5]:
# comprobación de que todas las columnas están presentes y no me he equivocado en ningún nombre
todos_presentes = all(item in df_espana.columns for item in columnas_interes_alimentacion)
todos_presentes

True

#### Pequeño dataset con posibles disruptores endocrinos (incompleto por ahora)

In [ ]:
# Aditivos
aditivos = [{"aditivo": "E127", "nombre_quimico": "Eritrosina", "otros_nombres": ["e127", "eritrosina", "rojo acido 51", "rojo 3", "rojo 14", "sal disódica de tetraiodo fluoresceína", "laca de eritrosina alumínica", "food red 14", "fd&c red 3", "acid red 51", "acid violet 51", "erythrosine b & erythrosin bs", "eritrosin", "eosina b", "eosin j", "ci 45430"], "tipo": "colorante sintético", "grupo": "colorantes", "DE_estado": "posible DE", "referencias": ["https://pmc.ncbi.nlm.nih.gov/articles/PMC11587131/#sec4-jox-14-00090"], "regulacion_UE": "Frutas y hortalizas en conserva"},
            {"aditivo": "E102", "nombre_quimico": "Tartrazina", "otros_nombres": ["e102", "tartracina", "tartrazina", "amarillo ácido 23", "amarillo 5", "amarillo alimentario 4", "sal sódica de tartrazina", "tz", "tartrazine", "acid yellow 23", "fd&c yellow 5", "food yellow 4", "tartrazine sodium salt"], "tipo": "colorante sintético", "grupo": "colorantes", "DE_estado": "posible DE", "referencias": ["https://pmc.ncbi.nlm.nih.gov/articles/PMC11587131/#sec4-jox-14-00090"], "regulacion_UE": "Grupo III: Colorantes alimentarios con límites máximos"},
            {"aditivo": "E215", "nombre_quimico": "Etilparabeno sódico", "otros_nombres": [], "tipo": "conservante sintético", "grupo": "parabenos", "DE_estado": "posible DE", "referencias": ["https://pmc.ncbi.nlm.nih.gov/articles/PMC11587131/#sec4-jox-14-00090", "https://pmc.ncbi.nlm.nih.gov/articles/PMC10607526/?spm=a2ty_o01.29997173.0.0.6140c9210nkFNh"], "regulacion_UE": ""},
            {"aditivo": "E219", "nombre_quimico": "Metilparabeno sódico", "otros_nombres": [], "tipo": "conservante sintético", "grupo": "parabenos", "DE_estado": "posible DE", "referencias": ["https://pmc.ncbi.nlm.nih.gov/articles/PMC11587131/#sec4-jox-14-00090", "https://pmc.ncbi.nlm.nih.gov/articles/PMC10607526/?spm=a2ty_o01.29997173.0.0.6140c9210nkFNh"], "regulacion_UE": ""},
            {"aditivo": "E217", "nombre_quimico": "Propilparabeno sódico", "otros_nombres": [], "tipo": "conservante sintético", "grupo": "parabenos", "DE_estado": "posible DE", "referencias": ["https://pmc.ncbi.nlm.nih.gov/articles/PMC11587131/#sec4-jox-14-00090", "https://pmc.ncbi.nlm.nih.gov/articles/PMC10607526/?spm=a2ty_o01.29997173.0.0.6140c9210nkFNh"], "regulacion_UE": "no permitido"},
            {"aditivo": "E214", "nombre_quimico": "Etilparabeno", "otros_nombres": ["e214", "Etilhidroxibenzoato", "Etil-P-Hidroxibenzoato", "Etil-PHB", "Etilparabeno", "Ethylparaben"], "tipo": "conservante sintético", "grupo": "parabenos", "DE_estado": "posible DE", "referencias": ["https://pmc.ncbi.nlm.nih.gov/articles/PMC11587131/#sec4-jox-14-00090", "https://pmc.ncbi.nlm.nih.gov/articles/PMC10607526/?spm=a2ty_o01.29997173.0.0.6140c9210nkFNh"], "regulacion_UE": ""},
            {"aditivo": "E218", "nombre_quimico": "Metilparabeno", "otros_nombres": [], "tipo": "conservante sintético", "grupo": "parabenos", "DE_estado": "posible DE", "referencias": ["https://pmc.ncbi.nlm.nih.gov/articles/PMC11587131/#sec4-jox-14-00090", "https://pmc.ncbi.nlm.nih.gov/articles/PMC10607526/?spm=a2ty_o01.29997173.0.0.6140c9210nkFNh"], "regulacion_UE": ""},
            {"aditivo": "E216", "nombre_quimico": "Propilparabeno", "otros_nombres": [], "tipo": "conservante sintético", "grupo": "parabenos", "DE_estado": "posible DE", "referencias": ["https://pmc.ncbi.nlm.nih.gov/articles/PMC11587131/#sec4-jox-14-00090", "https://pmc.ncbi.nlm.nih.gov/articles/PMC10607526/?spm=a2ty_o01.29997173.0.0.6140c9210nkFNh"], "regulacion_UE": "no permitido"},
            {"aditivo": "E320", "nombre_quimico": "Butilhidroxianisol (BHA)", "otros_nombres": [], "tipo": "antioxidante sintético", "grupo": "antioxidantes", "DE_estado": "posible DE", "referencias": ["https://pubmed.ncbi.nlm.nih.gov/31710847/", "https://pmc.ncbi.nlm.nih.gov/articles/PMC4462476/?spm=a2ty_o01.29997173.0.0.6140c9210nkFNh"], "regulacion_UE": ""},
            {"aditivo": "E321", "nombre_quimico": "Butilhidroxitolueno (BHT)", "otros_nombres": [], "tipo": "antioxidante sintético", "grupo": "antioxidantes", "DE_estado": "posible DE", "referencias": ["https://pmc.ncbi.nlm.nih.gov/articles/PMC6728168/#S22"], "regulacion_UE": ""},
            ]

In [12]:
aditivos_df = pd.DataFrame(aditivos)

Muestra mediante un head() los principales datasets con los que vas a trabajar

In [6]:
# Primer dataset, sin haber seleccionado aún columnas de interés
df_espana.head()

,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,last_modified_by,last_updated_t,last_updated_datetime,product_name,abbreviated_product_name,generic_name,quantity,packaging,packaging_tags,packaging_en,packaging_text,brands,brands_tags,brands_en,categories,categories_tags,categories_en,origins,origins_tags,origins_en,manufacturing_places,manufacturing_places_tags,labels,labels_tags,labels_en,emb_codes,emb_codes_tags,first_packaging_code_geo,cities,cities_tags,purchase_places,stores,countries,countries_tags,countries_en,ingredients_text,ingredients_tags,ingredients_analysis_tags,allergens,allergens_en,traces,traces_tags,traces_en,serving_size,serving_quantity,no_nutrition_data,additives_n,additives,additives_tags,additives_en,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,food_groups,food_groups_tags,food_groups_en,states,states_tags,states_en,brand_owner,environmental_score_score,environmental_score_grade,nutrient_levels_tags,product_quantity,owner,data_quality_errors_tags,unique_scans_n,popularity_tags,completeness,last_image_t,last_image_datetime,main_category,main_category_en,image_url,image_small_url,image_ingredients_url,image_ingredients_small_url,image_nutrition_url,image_nutrition_small_url,energy-kj_100g,energy-kcal_100g,energy_100g,energy-from-fat_100g,fat_100g,saturated-fat_100g,butyric-acid_100g,caproic-acid_100g,caprylic-acid_100g,capric-acid_100g,lauric-acid_100g,myristic-acid_100g,palmitic-acid_100g,stearic-acid_100g,arachidic-acid_100g,behenic-acid_100g,lignoceric-acid_100g,cerotic-acid_100g,montanic-acid_100g,melissic-acid_100g,unsaturated-fat_100g,monounsaturated-fat_100g,omega-9-fat_100g,polyunsaturated-fat_100g,omega-3-fat_100g,omega-6-fat_100g,alpha-linolenic-acid_100g,eicosapentaenoic-acid_100g,docosahexaenoic-acid_100g,linoleic-acid_100g,arachidonic-acid_100g,gamma-linolenic-acid_100g,dihomo-gamma-linolenic-acid_100g,oleic-acid_100g,elaidic-acid_100g,gondoic-acid_100g,mead-acid_100g,erucic-acid_100g,nervonic-acid_100g,trans-fat_100g,cholesterol_100g,carbohydrates_100g,sugars_100g,added-sugars_100g,sucrose_100g,glucose_100g,fructose_100g,galactose_100g,lactose_100g,maltose_100g,maltodextrins_100g,psicose_100g,starch_100g,polyols_100g,erythritol_100g,isomalt_100g,maltitol_100g,sorbitol_100g,fiber_100g,soluble-fiber_100g,polydextrose_100g,insoluble-fiber_100g,proteins_100g,casein_100g,serum-proteins_100g,nucleotides_100g,salt_100g,added-salt_100g,sodium_100g,alcohol_100g,vitamin-a_100g,beta-carotene_100g,vitamin-d_100g,vitamin-e_100g,vitamin-k_100g,vitamin-c_100g,vitamin-b1_100g,vitamin-b2_100g,vitamin-pp_100g,vitamin-b6_100g,vitamin-b9_100g,folates_100g,vitamin-b12_100g,biotin_100g,pantothenic-acid_100g,silica_100g,bicarbonate_100g,potassium_100g,chloride_100g,calcium_100g,phosphorus_100g,iron_100g,magnesium_100g,zinc_100g,copper_100g,manganese_100g,fluoride_100g,selenium_100g,chromium_100g,molybdenum_100g,iodine_100g,caffeine_100g,taurine_100g,methylsulfonylmethane_100g,hydroxymethylbutyrate_100g,ph_100g,fruits-vegetables-legumes_100g,collagen-meat-protein-ratio_100g,cocoa_100g,chlorophyl_100g,carbon-footprint_100g,glycemic-index_100g,water-hardness_100g,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g,water_100g
0,9800300037,https://world-en.openfoodfacts.org/product/000...,kiliweb,1534753337,2018-08-20T08:22:17Z,1786787931,2026-08-15T09:58:51Z,chevalstar,1790162752,2026-09-23T11:25:52Z,gum cool watermelon,NaN,NaN,56 Pieces,NaN,NaN,NaN,NaN,Tic Tac,xx:tic-tac,tic-tac,NaN,"en:snacks,en:sweet-snacks,en:confectioneries,e...","Snacks,Sweet snacks,Confectioneries,en:chewing...",NaN,NaN,NaN,NaN,NaN,NaN,en:made-in-italy,Made in Italy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"en:france,en:spain,en:united-states","France,Spain,United States","XYLITOL, GUM BASE, ISOMALT, GUM ARABIC, SORBIT...","en:e967,en:gum-base,en:e953,en:e414,en:e420,en...","en:palm-oil-free,en:vegan-s

In [13]:
# Siguiente dataset...
aditivos_df.head()

,aditivo,nombre_quimico,otros_nombres,tipo,grupo,DE_estado,referencias,regulacion_UE
0,E127,Eritrosina,"[e127, eritrosina, rojo acido 51, rojo 3, rojo...",colorante sintético,colorantes,posible DE,[https://pmc.ncbi.nlm.nih.gov/articles/PMC1158...,Frutas y hortalizas en conserva
1,E102,Tartrazina,"[e102, tartracina, tartrazina, amarillo ácido ...",colorante sintético,colorantes,posible DE,[https://pmc.ncbi.nlm.nih.gov/articles/PMC1158...,Grupo III: Colorantes alimentarios con límites...
2,E215,,[],conservante sintético,parabenos,posible DE,[https://pmc.ncbi.nlm.nih.gov/articles/PMC1158...,
3,E219,,[],conservante sintético,parabenos,posible DE,[https://pmc.ncbi.nlm.nih.gov/articles/PMC1158...,
4,E217,,[],conservante sintético,parabenos,posible DE,[https://pmc.ncbi.nlm.nih.gov/articles/PMC1158...,
